# Qwen3-4B experiments on Colab

Choose **Runtime → Change runtime type → T4 GPU or A100 GPU**, then run the cells in order. Approve Google Drive access when prompted. T4 uses FP16 and A100 uses BF16. Each run loads one model and reuses the repository's single-agent, ensemble, and hierarchy solvers.

The defaults reproduce the existing length-2 and length-4 calibration pairs, with a 4,000-token cap per problem per system including all prompts and generated reasoning. The settings cell can generate more pairs.

Results are saved after every completed problem under `MyDrive/small-multi-agents/runs/`. To resume after a disconnect, reconnect with the same GPU type and rerun the notebook with the same settings and run name. A partially completed problem runs again. Change `RUN_NAME` when changing the GPU, settings, or code version. Precision and software versions are recorded with the run because sampled answers can differ between environments.

In [1]:
# Get the repository and install its inference dependencies.
import os
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/content/small-multi-agents")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/liatzheng/small-multi-agents.git", str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Keep Colab's preinstalled CUDA PyTorch and record its version below.
requirements = [
    line.strip() for line in Path("requirements.txt").read_text().splitlines()
    if line.strip() and not line.startswith(("#", "torch=="))
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Repository", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Repository 6fab517568027ee12e89b7e0c5d5f0a3a93f4780


In [2]:
# Check the GPU, then connect persistent result storage.
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Choose a T4 or A100 GPU in Runtime → Change runtime type, then rerun.")
print("GPU", torch.cuda.get_device_name(0))
print("GPU memory (GiB)", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch", torch.__version__)
drive.mount("/content/drive")

GPU Tesla T4
GPU memory (GiB) 14.6
PyTorch 2.14.1+cu130
Mounted at /content/drive


In [3]:
# Experiment settings. The defaults match the saved calibration problems.
RUN_NAME = "calibration-4k"
CHAIN_LENGTHS = [2, 4]
PAIRS_PER_LENGTH = 1
PROBLEM_SEED = 2027
SAMPLING_SEED = 42
BUDGET = 4000

import hashlib
import json
import tempfile
import transformers
import accelerate
from generate_problems import generate_chain_problems
from run_single_agent import MODEL, REVISION

if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in (".", ".."):
    raise ValueError("Choose a simple folder name for RUN_NAME.")
if not CHAIN_LENGTHS or len(set(CHAIN_LENGTHS)) != len(CHAIN_LENGTHS):
    raise ValueError("CHAIN_LENGTHS must contain distinct positive lengths.")
if not 1 <= BUDGET <= 32768:
    raise ValueError("BUDGET must be between 1 and 32768.")

problems = [
    problem
    for length in CHAIN_LENGTHS
    for problem in generate_chain_problems(PAIRS_PER_LENGTH, PROBLEM_SEED, length)
]
problem_text = "".join(json.dumps(problem) + "\n" for problem in problems)
run_dir = Path("/content/drive/MyDrive/small-multi-agents/runs") / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
run_config = {
    "model": MODEL, "revision": REVISION,
    "budget": BUDGET, "sampling_seed": SAMPLING_SEED,
    "problem_seed": PROBLEM_SEED, "pairs_per_length": PAIRS_PER_LENGTH,
    "chain_lengths": CHAIN_LENGTHS,
    "problems_sha256": hashlib.sha256(problem_text.encode()).hexdigest(),
    "gpu": torch.cuda.get_device_name(0), "dtype": str(dtype),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}

def save_text(path, text):
    # Publish a complete file so an interrupted write can be retried on resume.
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as stream:
        temporary_path = Path(stream.name)
        stream.write(text)
        stream.flush()
        os.fsync(stream.fileno())
    temporary_path.replace(path)

config_path = run_dir / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != run_config:
    raise ValueError("This run has different settings or hardware. Choose a new RUN_NAME.")
save_text(config_path, json.dumps(run_config, indent=2) + "\n")
save_text(run_dir / "problems.jsonl", problem_text)
print(f"{len(problems)} problems × 3 systems = {len(problems) * 3} evaluations")
print("Precision", dtype)
print("Results", run_dir)

4 problems × 3 systems = 12 evaluations
Precision torch.float16
Results /content/drive/MyDrive/small-multi-agents/runs/calibration-4k


In [4]:
# Load Qwen once for all systems. The first load downloads about 8 GB.
import gc
from run_single_agent import load_model, solve
from run_ensemble import solve_ensemble
from run_hierarchy import solve_hierarchy

if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()
model, tokenizer = load_model()
print("Loaded", MODEL, "on", model.device, "with", model.dtype)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

... (4 outputs left out)

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-4B on cuda:0 with torch.float16


In [5]:
# Run all three systems. Each completed problem gets a checkpoint.
from time import perf_counter

solvers = {"single_agent": solve, "ensemble": solve_ensemble, "hierarchy": solve_hierarchy}
records = []
for system, solver in solvers.items():
    for problem in problems:
        label = f"{system}_d{problem['chain_length']}_{problem['pair_id']}_{problem['kind']}"
        checkpoint = run_dir / f"{label}.json"
        if checkpoint.exists():
            record = json.loads(checkpoint.read_text())
            expected = {
                **problem, "system": system, "budget": BUDGET,
                "model": MODEL, "revision": REVISION, "dtype": str(model.dtype), "seed": SAMPLING_SEED,
            }
            if any(record.get(key) != value for key, value in expected.items()):
                raise ValueError(f"Checkpoint differs from this run: {checkpoint}")
            print("Resumed", label, record["status"], flush=True)
        else:
            print("Running", label, flush=True)
            started = perf_counter()
            record = solver(problem, model, tokenizer, BUDGET, SAMPLING_SEED)
            record.update(
                system=system, model=MODEL, revision=REVISION, seed=SAMPLING_SEED,
                seconds=round(perf_counter() - started, 3),
            )
            if record["total_tokens"] > BUDGET:
                raise RuntimeError(f"Token budget exceeded for {label}")
            save_text(checkpoint, json.dumps(record, indent=2) + "\n")
            print(f"Saved {record['status']}, {record['total_tokens']} tokens", flush=True)
        records.append(record)
    system_records = [record for record in records if record["system"] == system]
    save_text(
        run_dir / f"{system}.jsonl",
        "".join(json.dumps(record) + "\n" for record in system_records),
    )
print("All evaluations complete.")

Resumed single_agent_d2_1_independent correct
Running single_agent_d2_1_dependent
Saved correct, 3454 tokens
Running single_agent_d4_1_independent
Saved correct, 3587 tokens
Running single_agent_d4_1_dependent
Saved length_limit, 4000 tokens
Running ensemble_d2_1_independent
Saved no_majority, 4000 tokens
Running ensemble_d2_1_dependent
Saved no_majority, 4000 tokens
Running ensemble_d4_1_independent
Saved no_majority, 4000 tokens
Running ensemble_d4_1_dependent
Saved no_majority, 4000 tokens
Running hierarchy_d2_1_independent
Saved length_limit, 1000 tokens
Running hierarchy_d2_1_dependent
Saved length_limit, 1000 tokens
Running hierarchy_d4_1_independent
Saved length_limit, 1000 tokens
Running hierarchy_d4_1_dependent
Saved length_limit, 1000 tokens
All evaluations complete.


In [6]:
# Compare accuracy and mean token usage for each length and problem kind.
import pandas as pd
from IPython.display import display

frame = pd.DataFrame(records)
summary = frame.groupby(["chain_length", "kind", "system"], sort=True).agg(
    problems=("correct", "size"),
    correct=("correct", "sum"),
    accuracy=("correct", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
).reset_index()
failures = frame.groupby(["chain_length", "kind", "system", "status"]).size().rename("problems").reset_index()
save_text(run_dir / "summary.csv", summary.to_csv(index=False))
save_text(run_dir / "status_counts.csv", failures.to_csv(index=False))
display(summary.round(3))
display(failures)
print("Saved result logs, summaries, problem inputs, and run configuration to", run_dir)

,chain_length,kind,system,problems,correct,accuracy,mean_input_tokens,mean_output_tokens,mean_total_tokens
0,2,dependent,ensemble,1,0,0.0,618.0,3382.0,4000.0
1,2,dependent,hierarchy,1,0,0.0,264.0,736.0,1000.0
2,2,dependent,single_agent,1,1,1.0,206.0,3248.0,3454.0
3,2,independent,ensemble,1,0,0.0,618.0,3382.0,4000.0
4,2,independent,hierarchy,1,0,0.0,264.0,736.0,1000.0
5,2,independent,single_agent,1,1,1.0,206.0,2751.0,2957.0
6,4,dependent,ensemble,1,0,0.0,975.0,3025.0,4000.0
7,4,dependent,hierarchy,1,0,0.0,383.0,617.0,1000.0
8,4,dependent,single_agent,1,0,0.0,325.0,3675.0,4000.0
9,4,independent,ensemble,1,0,0.0,975.0,3025.0,4000.0


,chain_length,kind,system,status,problems
0,2,dependent,ensemble,no_majority,1
1,2,dependent,hierarchy,length_limit,1
2,2,dependent,single_agent,correct,1
3,2,independent,ensemble,no_majority,1
4,2,independent,hierarchy,length_limit,1
5,2,independent,single_agent,correct,1
6,4,dependent,ensemble,no_majority,1
7,4,dependent,hierarchy,length_limit,1
8,4,dependent,single_agent,length_limit,1
9,4,independent,ensemble,no_majority,1


Saved result logs, summaries, problem inputs, and run configuration to /content/drive/MyDrive/small-multi-agents/runs/calibration-4k


The per-problem JSON files preserve progress during interruptions. The three JSONL files contain the model responses, scoring, and token counts for each system. `summary.csv` reports accuracy and mean token usage, and `status_counts.csv` includes truncation and protocol failures. These files can be added to the repository's `results/` directory for the paper.

For a larger pilot, change `PAIRS_PER_LENGTH` to `10` and choose a new `RUN_NAME`. This runs 40 problems across the three systems. Keep a comparison on the same GPU type and precision. Once finished, use **Runtime → Disconnect and delete runtime** to release the GPU. The result files remain in Drive.